# Historical rice-cycle and fire-region maps

Two exploratory maps show southern Vietnam rice cycles and selected regional fire-source polygons.
The embedded maps and table previews come from revision `19430e7`; their use in the final manuscript is unconfirmed.

## Setup and inputs

Use `pandas`, `numpy`, `geopandas`, `matplotlib`, and a GeoPandas-compatible spatial reader.
Exact package versions were not recorded.
Update the original Windows paths to your local files.

| Input | Purpose |
|---|---|
| `datasets/paper figures/SVNM_rice_cycles.csv` | Longitude, latitude, plot ID, and rice cycles |
| `shapefiles/gadm41_VNM_1.json` | Province boundaries and labels |
| `datasets/paper figures/FIRMS47_mapping.csv` | `ASIA=1, FRP=2` selection and `GID_2` |
| `shapefiles/gadm41_{country}_0.json` and `gadm41_{country}_2.json` | Country/admin-2 boundaries for THA, LAO, MYS, KHM, IDN, CHN, VNM, PHL |

The maps display existing classifications without computing crop calendars or fire instruments.
Points and boundaries must share a longitude/latitude reference system.
The point GeoDataFrame has no explicit CRS; geographic centroids approximate label positions.

Cleanup preserved plotting code and saved maps, cleared warnings and execution counts, and checked format/syntax without loading data.
After confirming inputs, rerun with `python -m jupyter nbconvert --execute --to notebook --inplace draw_paper_figures.ipynb`.


## 1. Import libraries

Uses local CSV and GeoJSON inputs.


In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap
import re

## 2. Load rice points and provinces

Requires `lon`, `lat`, `id`, and `cycles`.
Preserves province selection and labels; the saved table previews those boundaries.


In [ ]:
df = pd.read_csv(r"D:\Work\ADB\SAR\datasets\paper figures\SVNM_rice_cycles.csv")
geom = gpd.points_from_xy(df.lon, df.lat)
gdf = gpd.GeoDataFrame(df[['id', 'cycles']], geometry=geom)
poly = gpd.read_file(r"D:\Work\ADB\SAR\shapefiles\gadm41_VNM_1.json")
poly = poly[poly['VARNAME_1'].isin(['LongAn', 'DongThap', 'TienGiang', 'AnGiang', 'CanTho', 'VinhLong',
                                    'BenTre', 'KienGiang', 'HauGiang', 'TraVinh', 'SocTrang', 'BacLieu', 'CaMau',
                                    'BinhPhuoc', 'TayNinh', 'BinhDuong', 'DongNai', 'BaRia-VungTau', 'HoChiMinh'])]
poly['NAME_1_EN'] = poly['VARNAME_1'].apply(lambda x: re.sub(r'([a-z])([A-Z])', r'\1 \2', x))
# poly[poly['VARNAME_1'].str.contains('Phu')]
poly['centroid_x'] = poly.centroid.x
poly['centroid_y'] = poly.centroid.y
poly

### Inspect rice points

The saved table previews the input schema and coordinates.


In [ ]:
gdf

### Plot rice cycles

Green shades represent cycles 1, 2, and 3.
The legend assumes all three categories occur in that order; check changed inputs.


In [ ]:
# Create a custom colormap for discrete values
colors = ['#99EDC3', '#03C04A', '#3A5A40']  # Green, Yellow, Red for 1, 2, 3 cycles
cmap = ListedColormap(colors)

# Create custom legend labels
legend_labels = {1: 'Cycle 1', 2: 'Cycle 2', 3: 'Cycle 3'}

fig, ax = plt.subplots(figsize=(15, 15))
poly.boundary.plot(color='black', ax=ax)
for idx, row in poly.iterrows():
        ax.annotate(text=row['NAME_1_EN'],
                    xy=(row['centroid_x'], row['centroid_y']),
                    xytext=(3, 3),  # Offset text slightly
                    textcoords="offset points",
                    ha='center',
                    fontsize=12,
                    color='black')

# Plot with discrete colormap
gdf.plot(column='cycles', cmap=cmap, categorical=True, legend=True, ax=ax, markersize=2,
         legend_kwds={'title': 'Rice Cropping Pattern', 'loc': 'upper left'})

# Modify the legend labels and font sizes after plotting
legend = ax.get_legend()
# Increase title font size
legend.get_title().set_fontsize(14)  # Adjust title font size
# Modify labels and their font size
for i, text in enumerate(legend.get_texts()):
    text.set_text(legend_labels[i + 1])  # i + 1 because our values are 1-based
    text.set_fontsize(12)  # Adjust label font size

ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])
# plt.title("Study Area and Rice Cropping Cycles", fontsize=20)



The saved map shows the input cycle classification.


## 3. Load fire regions

Uppercases columns and splits on `ASIA=1, FRP=2`.
The preview shows `asia`; the map uses `frp`.


In [ ]:
df = pd.read_csv(r"D:\Work\ADB\SAR\datasets\paper figures\FIRMS47_mapping.csv")
df.columns = df.columns.str.upper()
asia = df[df['ASIA=1, FRP=2'] == 1]
frp = df[df['ASIA=1, FRP=2'] == 2]

asia.head()


### Load country boundaries

Both loops use the eight-country list above, which differs from the later manuscript source-country specification.


In [ ]:
polys = []
for id in ['THA', 'LAO', 'MYS', 'KHM', 'IDN', 'CHN', 'VNM', 'PHL']:
    poly = gpd.read_file(rf"D:\Work\ADB\SAR\shapefiles\gadm41_{id}_0.json")
    polys.append(poly)

poly_df0 = gpd.GeoDataFrame(pd.concat(polys, axis=0).reset_index(drop=True))


polys = []
for id in ['THA', 'LAO', 'MYS', 'KHM', 'IDN', 'CHN', 'VNM', 'PHL']:
    poly = gpd.read_file(rf"D:\Work\ADB\SAR\shapefiles\gadm41_{id}_2.json")
    polys.append(poly)
poly_df2 = gpd.GeoDataFrame(pd.concat(polys, axis=0).reset_index(drop=True))

### Join fire regions

Left-joins `GID_2` to GADM admin-2 polygons.
Check unmatched geometries and duplicate IDs when changing inputs.
An alternative join remains commented.


In [ ]:
# asia = gpd.GeoDataFrame(pd.merge(asia, poly_df, on='GID_2', how='left'))
# asia = asia[asia['geometry'].notna()]
frp = gpd.GeoDataFrame(pd.merge(frp, poly_df2, on='GID_2', how='left'))



### Plot fire regions

Grey outlines show countries; aqua shows selected regions.
Labels use geographic centroids with Vietnam/Philippines offsets.
Colors indicate membership, not fire intensity or effects.


In [ ]:
poly_df0['centroid_x'] = poly_df0.centroid.x
poly_df0['centroid_y'] = poly_df0.centroid.y
poly_df0 = poly_df0.drop_duplicates(subset=['COUNTRY'], keep='first')

fig, ax = plt.subplots(figsize=(15, 15))
poly_df0.plot(ax=ax, linewidth=0.5, color='grey', alpha=0.2)
poly_df0.boundary.plot(ax=ax, linewidth=0.5, color='black', alpha=1)
for idx, row in poly_df0.iterrows():
        if row['COUNTRY'] == 'Philippines':
                ax.annotate(text=row['COUNTRY'],
                    xy=(row['centroid_x'], row['centroid_y']),
                    xytext=(50, 30),  # Offset text slightly
                    textcoords="offset points",
                    ha='center',
                    fontsize=12,
                    color='red')
        elif row['COUNTRY'] == 'Vietnam':
                ax.annotate(text=row['COUNTRY'],
                    xy=(row['centroid_x'], row['centroid_y']),
                    xytext=(30, 0),  # Offset text slightly
                    textcoords="offset points",
                    ha='center',
                    fontsize=12,
                    color='red')
        else:
                ax.annotate(text=row['COUNTRY'],
                    xy=(row['centroid_x'], row['centroid_y']),
                    xytext=(0, 0),  # Offset text slightly
                    textcoords="offset points",
                    ha='center',
                    fontsize=12,
                    color='red')



frp.plot(ax=ax, linewidth=1, color='aqua', alpha=1)

ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])

## Before publication

Figures remain embedded but the code does not save named manuscript files.
Confirm geography, labels, and figure numbers against the manuscript.
